# 04 — Limitation experiment: does the evaluator's score depend on the register?

No GPU needed. This notebook reads every `results/scores_*.csv` produced by notebooks 02, 03 and 05 and measures register bias in a single, evaluator-independent way.

## The logic of the experiment
For the **correct** answers, the four registers carry *identical information* (verified in notebook 01), so a fair evaluator must give the same score to all four. Any systematic difference is **register bias**. Everything is *paired by question*: we compare the MSA, Egyptian, Franco and code-switched version of the same answer.

## Measures (all in `src/common.py`)
| Measure | Meaning |
|---|---|
| `quality_gap` | mean score of correct answers minus mean score of incorrect answers: the evaluator's own yardstick |
| `norm_gap` | register gap divided by `quality_gap`. Example: `+0.30` means "writing the answer in this register costs 30% of the difference between a correct and an incorrect answer". Makes cosine, BERTScore and 1-5 scores comparable |
| `p_holm` | paired Wilcoxon signed-rank test over the 30 questions, Holm-corrected over all register pairs |
| bootstrap CI | 95% interval of the mean gap (10,000 resamples, seed fixed) |
| `friedman_p` | omnibus test: do the 4 registers differ at all? |
| `inconsistency_norm` | mean per-question standard deviation across the 4 registers, divided by `quality_gap` (0 = perfectly register-invariant) |
| flip rate | how often a **correct** answer in register A scores below an **incorrect** answer in register B (same question). This is the decision-level harm of bias: a right answer losing to a wrong one because of how it is written |
| `spearman_*` | agreement with the ground truth, overall and inside each register. A fair judge must stay accurate in **every** register |

**Ground truth.** `gold` is the mean human rating when `data/human_ratings.csv` exists; otherwise a provisional proxy from the quality level (printed below). Re-run this notebook after annotation is finished.

In [ ]:
import os, sys
REPO_DIR = "register_bias_judge"
if not os.path.exists("src/common.py"):
    if os.path.isdir(REPO_DIR):                                   # unzipped next to the notebook
        os.chdir(REPO_DIR)
    else:
        try:                                                      # Google Colab: the repo lives in your Drive, so
            from google.colab import drive                        # data/ and results/ survive between sessions
            drive.mount("/content/drive")
            os.chdir(f"/content/drive/MyDrive/{REPO_DIR}")
        except Exception as e:
            print("Could not locate the repo folder:", e)
sys.path.insert(0, os.getcwd())
assert os.path.exists("src/common.py"), "Folder 'register_bias_judge' not found. Put it in the root of your Google Drive (My Drive) and re-run."
print("working in:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import common as C

C.set_seed(C.SEED)
items = C.load_items()
pairs = C.build_pairs(items)
print("ground truth source:", pairs.gold_source.value_counts().to_dict())

sc = C.load_scores()
family = sc.drop_duplicates("evaluator").set_index("evaluator").family
df = sc.merge(pairs.drop(columns=["gold_source"]), on="pair_id", how="inner")

cov = df.groupby("evaluator").size()
bad = cov[cov != len(pairs)]
assert bad.empty, f"these evaluators do not cover all {len(pairs)} pairs (re-run their notebook):\n{bad}"
order = [e for fam in ["embedding", "bertscore", "llm", "lora"] for e in sorted(family[family == fam].index)]
order += [e for e in family.index if e not in order]
print(f"{len(order)} evaluators:", ", ".join(order))

## 1) Summary table (one row per evaluator x reference register)

In [ ]:
summ = C.summarize(df)
summ["family"] = summ.evaluator.map(family)
summ = summ.set_index("evaluator").loc[order].reset_index()
summ.to_csv(C.RESULTS / "bias_summary.csv", index=False)
cols = ["evaluator", "family", "ref_register", "quality_gap", "max_penalty_vs_msa_norm", "friedman_p",
        "inconsistency_norm", "flip_cross_register", "spearman_all"]
summ[cols].round(3)

## 2) Figure 1 — register gap relative to MSA

Each cell is `norm_gap` = (score of MSA − score of the register) / quality_gap. **Red = that register is penalised relative to MSA.** `*` marks a gap that is significant after Holm correction (p < 0.05). **n/a** = the evaluator does not rank correct above incorrect (`quality_gap` <= 0), so a gap cannot be expressed as a fraction of it; such evaluators are not measuring content and are reported separately below. The left panel uses an Egyptian reference, the right panel an MSA reference.

In [ ]:
pw = C.pairwise_register_gaps(df)
pw.to_csv(C.RESULTS / "pairwise_register_gaps.csv", index=False)
vs = C.gaps_vs_msa(pw)
cols3 = ["egyptian", "franco", "codeswitch"]

fig, axes = plt.subplots(1, 2, figsize=(11, 0.42 * len(order) + 2), sharey=True)
for ax, rr in zip(axes, ["egyptian", "msa"]):
    sub = vs[vs.ref_register == rr]
    tab = sub.pivot(index="evaluator", columns="other", values="norm_gap").reindex(order)[cols3]
    pt = sub.pivot(index="evaluator", columns="other", values="p_holm").reindex(order)[cols3]
    im = ax.imshow(tab.values.astype(float), cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels([C.REG_LABEL[c] for c in cols3])
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8)
    for i in range(len(order)):
        for j in range(3):
            v, p = tab.values[i, j], pt.values[i, j]
            ax.text(j, i, "n/a" if np.isnan(v) else f"{v:+.2f}{'*' if p < 0.05 else ''}", ha="center", va="center", fontsize=8)
    ax.set_title(f"reference = {C.REG_LABEL[rr]}")
fig.colorbar(im, ax=axes, shrink=0.6, label="penalty vs MSA (fraction of quality gap)")
fig.savefig(C.FIGS / "fig1_register_gap_vs_msa.png", dpi=200, bbox_inches="tight")
plt.show()

## 3) Figure 2 — flip matrices (Egyptian reference)

Rows: the register of a **correct** answer. Columns: the register of an **incorrect** answer to the same question. Each cell is the fraction of questions where the correct answer scores **below** the incorrect one (0.5 = tie). A fair judge is dark-free everywhere; a bright row (e.g. Franco) means correct answers written in that register lose to wrong answers.

In [ ]:
ncol = 4
nrow = int(np.ceil(len(order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.2 * nrow), squeeze=False)
short = [C.REG_LABEL[r][:4] for r in C.REGISTERS]
for ax, ev in zip(axes.ravel(), order):
    m = C.flip_matrix(df, ev, "egyptian")
    ax.imshow(m.values.astype(float), vmin=0, vmax=0.5, cmap="Reds")
    ax.set_xticks(range(4)); ax.set_xticklabels(short, fontsize=7)
    ax.set_yticks(range(4)); ax.set_yticklabels(short, fontsize=7)
    ax.set_title(ev, fontsize=8)
for ax in axes.ravel()[len(order):]:
    ax.axis("off")
fig.suptitle("P(correct answer in row-register scores below incorrect answer in column-register)", fontsize=9)
fig.savefig(C.FIGS / "fig2_flip_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

## 4) Figure 3 — validity inside each register

Spearman correlation with the ground truth computed **within** each register (90 answers each). A judge that is accurate on MSA but drops on Franco is unfair even when its *average* gap looks small.

In [ ]:
val = C.validity_table(df)
val.to_csv(C.RESULTS / "validity_by_register.csv", index=False)
v = val[val.ref_register == "egyptian"].set_index("evaluator").reindex(order)
ax = v[[f"spearman_{r}" for r in C.REGISTERS]].rename(columns=lambda c: C.REG_LABEL[c.replace("spearman_", "")]) \
      .plot.bar(figsize=(12, 4), width=0.85)
ax.set_ylabel("Spearman with ground truth"); ax.set_ylim(0, 1.05); ax.legend(ncol=4, fontsize=8)
plt.xticks(rotation=60, ha="right", fontsize=8); plt.tight_layout()
plt.savefig(C.FIGS / "fig3_validity_by_register.png", dpi=200, bbox_inches="tight")
plt.show()

## 5) Automatic findings (copy into the report after checking the figures)

A gap is reported only if it is **statistically significant** (Holm p < 0.05) **and practically relevant** (at least `MIN_EFFECT` = 5% of the evaluator's correct-vs-incorrect gap). With n = 30 and low-noise evaluators, tiny differences can be significant without mattering.

In [ ]:
MIN_EFFECT = 0.05
no_sep = summ[summ.quality_gap <= 0][["evaluator", "ref_register"]].drop_duplicates()
if len(no_sep):
    print("Excluded (do NOT rank correct above incorrect, so their bias cannot be interpreted):",
          ", ".join(sorted(set(no_sep.evaluator))))
    print()
sig = vs[(vs.p_holm < 0.05) & (vs.norm_gap >= MIN_EFFECT)].sort_values("norm_gap", ascending=False)
rev = vs[(vs.p_holm < 0.05) & (vs.norm_gap <= -MIN_EFFECT)].sort_values("norm_gap")
if sig.empty and rev.empty:
    print("No evaluator shows a significant and practically relevant register gap relative to MSA.")
for r in sig.itertuples():
    print(f"- {r.evaluator} [ref={r.ref_register}]: MSA scores higher than {C.REG_LABEL[r.other]} by {r.gap:.3f} "
          f"(95% CI {r.ci_lo:.3f} to {r.ci_hi:.3f}) = {100 * r.norm_gap:.0f}% of its correct-vs-incorrect gap; p_holm = {r.p_holm:.3g}")
for r in rev.itertuples():
    print(f"- (reverse) {r.evaluator} [ref={r.ref_register}]: {C.REG_LABEL[r.other]} scores HIGHER than MSA by {-r.gap:.3f} "
          f"= {-100 * r.norm_gap:.0f}% of its gap; p_holm = {r.p_holm:.3g}")

## 6) Does the proposed judge help?

Compare the LoRA judges (family `lora`) with the untuned base model and with the two LLM-judge baselines. What we hope to see for the research question: **lower** `max_penalty_vs_msa_norm`, `inconsistency_norm` and `flip_cross_register`, with `spearman_all` **not** lower.

In [ ]:
cmp = summ[summ.family.isin(["llm", "lora"]) & (summ.ref_register == "egyptian")]
cmp[["evaluator", "family", "max_penalty_vs_msa_norm", "inconsistency_norm", "flip_cross_register", "spearman_all"]].round(3)